# Computational Theory: SHA-256

## Problem 1: Representing SHA-256 Data

These examples follow the [SHA-256 standard](https://doi.org/10.6028/NIST.FIPS.180-4). I used [ChatGPT](https://chatgpt.com/) to help with this section.

### 32-bit words

I use [`np.uint32`](https://numpy.org/doc/stable/reference/arrays.scalars.html#unsigned-integer-types) because SHA-256 uses unsigned 32-bit words. Its range is 0 to 2**32 - 1; Python's normal `int` has no fixed size.

In [1]:
import numpy as np

word = np.uint32(0x12345678)
print("Decimal:", word)
print("Binary:", f"{word:032b}")
print("Hex:", f"{word:08x}")
print("Size in bytes:", word.itemsize)

Decimal: 305419896
Binary: 00010010001101000101011001111000
Hex: 12345678
Size in bytes: 4


### Sequences of words

A NumPy array with `dtype=np.uint32` keeps the words in order, with the same type for every element.

In [2]:
words = np.array([1, 2, 3], dtype=np.uint32)
print("Words:", words)
print("Type:", words.dtype)
print("Size in bytes:", words.nbytes)

Words: [1 2 3]
Type: uint32
Size in bytes: 12


### Input messages

I use `bytes` because the input is binary data. Text is [encoded as UTF-8](https://docs.python.org/3/library/stdtypes.html#str.encode); `café` has four characters but five bytes.

In [3]:
text = "café"
message = text.encode("utf-8")
print("Message:", message)
print("Characters:", len(text))
print("Bytes:", len(message))

Message: b'caf\xc3\xa9'
Characters: 4
Bytes: 5


### 512-bit message blocks

A block is 64 bytes, split into sixteen big-endian `uint32` words for processing. This is a sample block; padding is covered in Problem 4.

In [4]:
block = bytes(range(64))
block_words = np.array(
    [int.from_bytes(block[i:i + 4], "big") for i in range(0, 64, 4)],
    dtype=np.uint32,
)
print("Block bytes:", len(block))
print("Block words:", len(block_words))
print("First word:", f"{block_words[0]:08x}")
print("Last word:", f"{block_words[-1]:08x}")

Block bytes: 64
Block words: 16
First word: 00010203
Last word: 3c3d3e3f


### The final 256-bit hash

Eight `uint32` words make 32 bytes. I convert them to 64 lowercase hex characters for readable output. The example values only show the format.

In [5]:
hash_words = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=np.uint32)
digest = b"".join(int(word).to_bytes(4, "big") for word in hash_words)
hex_digest = digest.hex()
print("Digest bytes:", len(digest))
print("Hex digest:", hex_digest)
print("Hex characters:", len(hex_digest))

Digest bytes: 32
Hex digest: 0000000100000002000000030000000400000005000000060000000700000008
Hex characters: 64


### Big-endian integers

Big-endian puts the most significant byte first. I use [`int.from_bytes()` and `int.to_bytes()`](https://docs.python.org/3/library/stdtypes.html#int.from_bytes) with `"big"` to follow SHA-256's byte order.

In [6]:
value = 0x12345678
big_endian = value.to_bytes(4, "big")
little_endian = value.to_bytes(4, "little")
print("Big-endian:", big_endian.hex(" "))
print("Little-endian:", little_endian.hex(" "))
print("Read back:", hex(int.from_bytes(big_endian, "big")))

Big-endian: 12 34 56 78
Little-endian: 78 56 34 12
Read back: 0x12345678


## Problem 2: SHA-256 Bitwise Operations

The definitions follow section 4.1.2 of the [Secure Hash Standard](https://doi.org/10.6028/NIST.FIPS.180-4). I use `np.uint32` to keep operations within 32 bits. In Python, `&` means AND, `^` means XOR, and `~` means NOT.

### Ch: choose

`Ch(x, y, z) = (x & y) ^ (~x & z)`

At each bit position, `x` selects the bit from `y` when it is 1, otherwise from `z`. With `uint32`, NOT flips exactly 32 bits. The tests cover all eight combinations of all-zero and all-one words, a mixed pattern, and the highest bit.

In [7]:
def Ch(x, y, z):
    """Return a uint32 choosing bits from y or z using x."""
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return (x & y) ^ (~x & z)

In [8]:
from itertools import product

for x, y, z in product((0, 0xFFFFFFFF), repeat=3):
    result = Ch(x, y, z)
    assert result == (y if x else z)
    assert isinstance(result, np.uint32)

assert Ch(0xF0F0F0F0, 0xCCCCCCCC, 0xAAAAAAAA) == 0xCACACACA
assert Ch(0x80000000, 0xFFFFFFFF, 0) == 0x80000000
print("Ch tests passed")

Ch tests passed


### Maj: majority

`Maj(x, y, z) = (x & y) ^ (x & z) ^ (y & z)`

Each output bit is 1 when at least two input bits are 1. With three ones, XOR of the three pairwise results is also 1. The tests cover all eight combinations of all-zero and all-one words, a mixed pattern, and the highest bit.

In [9]:
def Maj(x, y, z):
    """Return the bitwise majority of three words as uint32."""
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return (x & y) ^ (x & z) ^ (y & z)

In [10]:
for x, y, z in product((0, 0xFFFFFFFF), repeat=3):
    result = Maj(x, y, z)
    expected = 0xFFFFFFFF if (x, y, z).count(0xFFFFFFFF) >= 2 else 0
    assert result == expected
    assert isinstance(result, np.uint32)

assert Maj(0xF0F0F0F0, 0xCCCCCCCC, 0xAAAAAAAA) == 0xE8E8E8E8
assert Maj(0x80000000, 0x80000000, 0) == 0x80000000
print("Maj tests passed")

Maj tests passed


### Right rotation

A right rotation moves bits from the right end to the left end. I use OR to join the two parts and `np.uint32` to keep 32 bits. SHA-256 uses rotation counts between 1 and 31.

In [11]:
def ROTR(x, n):
    """Rotate a 32-bit word right by 1 to 31 bits."""
    x = np.uint32(x)
    return (x >> np.uint32(n)) | (x << np.uint32(32 - n))

In [12]:
assert ROTR(0, 1) == 0
assert ROTR(0xFFFFFFFF, 1) == 0xFFFFFFFF
assert ROTR(1, 1) == 0x80000000
assert ROTR(1, 31) == 2
assert ROTR(0x12345678, 8) == 0x78123456
assert isinstance(ROTR(1, 1), np.uint32)

### Sigma0

`Sigma0(x) = ROTR(x, 2) ^ ROTR(x, 13) ^ ROTR(x, 22)`

I rotate the same word by 2, 13 and 22 bits, then combine the results with XOR. These rotation counts are defined in the SHA-256 standard.

In [13]:
def Sigma0(x):
    """Return the SHA-256 upper-case Sigma0 of x."""
    return ROTR(x, 2) ^ ROTR(x, 13) ^ ROTR(x, 22)

In [14]:
assert Sigma0(0) == 0
assert Sigma0(0xFFFFFFFF) == 0xFFFFFFFF
assert Sigma0(1) == 0x40080400
assert Sigma0(0x12345678) == 0x66146474
assert isinstance(Sigma0(1), np.uint32)

### Sigma1

`Sigma1(x) = ROTR(x, 6) ^ ROTR(x, 11) ^ ROTR(x, 25)`

This works like `Sigma0`, but rotates by 6, 11 and 25 bits. Both functions are used in the compression rounds.

In [15]:
def Sigma1(x):
    """Return the SHA-256 upper-case Sigma1 of x."""
    return ROTR(x, 6) ^ ROTR(x, 11) ^ ROTR(x, 25)

In [16]:
assert Sigma1(0) == 0
assert Sigma1(0xFFFFFFFF) == 0xFFFFFFFF
assert Sigma1(1) == 0x04200080
assert Sigma1(0x12345678) == 0x3561ABDA
assert isinstance(Sigma1(1), np.uint32)

### sigma0

`sigma0(x) = ROTR(x, 7) ^ ROTR(x, 18) ^ (x >> 3)`

I rotate `x` right by 7 and 18 bits and shift it right by 3 bits. Then I XOR the three results. The shift adds zeros on the left instead of wrapping bits around.

In [17]:
def sigma0(x):
    """Return sigma0 of a 32-bit word."""
    x = np.uint32(x)
    return ROTR(x, 7) ^ ROTR(x, 18) ^ (x >> np.uint32(3))

In [18]:
assert sigma0(0) == 0
assert sigma0(0xFFFFFFFF) == 0x1FFFFFFF
assert sigma0(1) == 0x02004000
assert sigma0(0x12345678) == 0xE7FCE6EE
assert isinstance(sigma0(1), np.uint32)

### sigma1

`sigma1(x) = ROTR(x, 17) ^ ROTR(x, 19) ^ (x >> 10)`

This does the same steps with different amounts: rotate right by 17 and 19 bits, shift right by 10 bits, then XOR the results.

In [19]:
def sigma1(x):
    """Return sigma1 of a 32-bit word."""
    x = np.uint32(x)
    return ROTR(x, 17) ^ ROTR(x, 19) ^ (x >> np.uint32(10))

In [20]:
assert sigma1(0) == 0
assert sigma1(0xFFFFFFFF) == 0x003FFFFF
assert sigma1(1) == 0x0000A000
assert sigma1(0x12345678) == 0xA1F78649
assert isinstance(sigma1(1), np.uint32)

## Problem 3: Generating the SHA-256 Constants

### Prime numbers

A prime number is greater than 1 and has no divisors except 1 and itself. I start at 2 and check each number against the primes already found. `all()` is true when none of them divides the number. I stop after finding `n` primes. This follows the approach suggested in the [course notes](https://github.com/ianmcloughlin/computational-theory/blob/main/notes/prime-numbers.ipynb).

In [21]:
def primes(n):
    """Return the first n prime numbers."""
    numbers = []
    candidate = 2
    while len(numbers) < n:
        if all(candidate % p != 0 for p in numbers):
            numbers.append(candidate)
        candidate += 1
    return numbers

In [22]:
assert primes(0) == []
assert primes(1) == [2]
assert primes(10) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
assert len(primes(64)) == 64
assert primes(64)[-1] == 311

### Cube roots

I find the cube roots of the first 64 primes with `np.cbrt()`. Then I subtract the whole part with `np.floor()` to keep only the part after the decimal point. I print the first five results.

In [23]:
roots = np.cbrt(primes(64))
fractions = roots - np.floor(roots)
print(fractions[:5])

[0.25992105 0.44224957 0.70997595 0.91293118 0.22398009]


### Round constants

I multiply each fraction by `2**32` and convert it to `np.uint32`. This drops the decimal part and gives a 32-bit word. I print each word as eight hex digits, following section 4.2.2 of the [SHA-256 standard](https://doi.org/10.6028/NIST.FIPS.180-4).

In [24]:
constants = (fractions * 2**32).astype(np.uint32)
for value in constants:
    print(f"{value:08x}")

428a2f98
71374491
b5c0fbcf
e9b5dba5
3956c25b
59f111f1
923f82a4
ab1c5ed5
d807aa98
12835b01
243185be
550c7dc3
72be5d74
80deb1fe
9bdc06a7
c19bf174
e49b69c1
efbe4786
0fc19dc6
240ca1cc
2de92c6f
4a7484aa
5cb0a9dc
76f988da
983e5152
a831c66d
b00327c8
bf597fc7
c6e00bf3
d5a79147
06ca6351
14292967
27b70a85
2e1b2138
4d2c6dfc
53380d13
650a7354
766a0abb
81c2c92e
92722c85
a2bfe8a1
a81a664b
c24b8b70
c76c51a3
d192e819
d6990624
f40e3585
106aa070
19a4c116
1e376c08
2748774c
34b0bcb5
391c0cb3
4ed8aa4a
5b9cca4f
682e6ff3
748f82ee
78a5636f
84c87814
8cc70208
90befffa
a4506ceb
bef9a3f7
c67178f2


### Checking the constants

I copied the 64 values from section 4.2.2 of the [standard](https://doi.org/10.6028/NIST.FIPS.180-4). I compare them with my results in the same order. All 64 values match.

In [25]:
expected = np.array([
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
    0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
    0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
    0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
    0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
    0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
    0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
    0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
    0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
], dtype=np.uint32)

assert np.array_equal(constants, expected)
print("All 64 match")

All 64 match


## Problem 4: Padding and Parsing Messages

## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256